# FastEdit - Colab Tutorial

Editing large language models within 10 seconds using ROME algorithm.

**Requirements:**
- Colab Pro/Pro+ with A100 GPU (40GB VRAM) for 7B/13B models
- Or use smaller models on free tier

## 1. Installation

In [ ]:
# Option 1: Install from PyPI
!pip install pyfastedit -q

# Option 2: Install from source (uncomment below)
# !git clone https://github.com/hiyouga/FastEdit.git
# %cd FastEdit
# !pip install -r requirements.txt -q

In [ ]:
# Install huggingface_hub for uploading
!pip install huggingface_hub -q

In [ ]:
# Check GPU
!nvidia-smi

## 2. Prepare Edit Data

Format:
- `prompt`: Template with `{}` placeholder for subject
- `subject`: The entity to edit knowledge about
- `target`: New knowledge to inject
- `queries`: (Optional) Test sentences for evaluation

In [ ]:
import json

# Example: Update outdated knowledge
edit_data = [
    {
        "prompt": "The prime minister of the {} is",
        "subject": "UK",
        "target": "Rishi Sunak",
        "queries": [
            "The prime minister of the United Kingdom is",
            "The name of prime minister of the UK is"
        ]
    },
    {
        "prompt": "The CEO of {} is",
        "subject": "Twitter",
        "target": "Elon Musk",
        "queries": [
            "Who runs Twitter?",
            "Twitter's CEO is"
        ]
    }
]

with open("edit_data.json", "w", encoding="utf-8") as f:
    json.dump(edit_data, f, indent=2, ensure_ascii=False)

print("Created edit_data.json")
print(json.dumps(edit_data, indent=2))

## 3. Run Model Editing & Save

### Supported Models & Configs

| Model | Config | VRAM |
|-------|--------|------|
| EleutherAI/gpt-j-6b | gpt-j-6b | 24GB |
| meta-llama/Llama-2-7b-hf | llama-7b | 24GB |
| meta-llama/Llama-2-13b-hf | llama-13b | 32GB |
| Qwen/Qwen3-8B | qwen3-8b | 24GB |
| Qwen/Qwen3-30B-A3B | qwen3-30b-a3b | 24GB |

In [ ]:
# Edit GPT-J-6B and SAVE to ./edited_model
# Use --output to save the edited model
!python -m fastedit.editor \
    --data edit_data.json \
    --model EleutherAI/gpt-j-6b \
    --config gpt-j-6b \
    --template default \
    --output ./edited_model

In [ ]:
# Alternative: Edit Qwen3-8B and save
# !python -m fastedit.editor \
#     --data edit_data.json \
#     --model Qwen/Qwen3-8B \
#     --config qwen3-8b \
#     --template default \
#     --output ./edited_qwen3

In [ ]:
# Verify saved model
!ls -la ./edited_model

## 4. Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login

# Login to Hugging Face (need write token)
# Get token from: https://huggingface.co/settings/tokens
notebook_login()

In [ ]:
from huggingface_hub import HfApi

# Upload edited model to Hugging Face
api = HfApi()

# Change these values
REPO_NAME = "your-username/gpt-j-6b-edited"  # Your repo name
LOCAL_PATH = "./edited_model"  # Path to saved model

# Create repo (if not exists)
api.create_repo(repo_id=REPO_NAME, exist_ok=True)

# Upload all files
api.upload_folder(
    folder_path=LOCAL_PATH,
    repo_id=REPO_NAME,
    repo_type="model"
)

print(f"Model uploaded to: https://huggingface.co/{REPO_NAME}")

In [ ]:
# Alternative: Use push_to_hub from transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the edited model
model = AutoModelForCausalLM.from_pretrained("./edited_model")
tokenizer = AutoTokenizer.from_pretrained("./edited_model")

# Push to hub
REPO_NAME = "your-username/gpt-j-6b-edited"
model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"Done! https://huggingface.co/{REPO_NAME}")

## 5. Test the Edited Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load edited model from local or HuggingFace
model_path = "./edited_model"  # or "your-username/gpt-j-6b-edited"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    torch_dtype=torch.float16,
    device_map="auto"
)

# Generate text
def generate(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=max_length, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test edited knowledge
print(generate("The prime minister of the UK is"))
print(generate("The CEO of Twitter is"))

## 6. Custom Edit Example (Vietnamese)

In [ ]:
# Vietnamese knowledge editing example
vn_edit_data = [
    {
        "prompt": "Thu do cua {} la",
        "subject": "Viet Nam",
        "target": "Ha Noi",
        "queries": [
            "Thu do Viet Nam la gi?",
            "Viet Nam co thu do la"
        ]
    },
    {
        "prompt": "Chu tich nuoc {} la",
        "subject": "Viet Nam",
        "target": "Luong Cuong",
        "queries": [
            "Ai la chu tich nuoc Viet Nam?"
        ]
    }
]

with open("vn_edit_data.json", "w", encoding="utf-8") as f:
    json.dump(vn_edit_data, f, indent=2, ensure_ascii=False)

print("Created vn_edit_data.json")